In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from scipy.stats import uniform, randint

In [2]:
# Datasets
datasets = {
    "heart": ("Data/heart_train.csv", "Data/heart_test.csv"),
    "diabetes": ("Data/diabetes_train.csv", "Data/diabetes_test.csv"),
    "cancer": ("Data/cancer_train.csv", "Data/cancer_test.csv"),
    "alzheimer": ("Data/alzheimer_train.csv", "Data/alzheimer_test.csv")
}

# 1. Uniform Random

In [3]:
# Grid of hyperparameters 
param_uniform = {
    'max_depth': randint(3, 15),                    # tree depth
    'learning_rate': uniform(0.01, 0.29),           # eta: 0.01-0.3
    'n_estimators': randint(50, 500),               # number of trees
    'subsample': uniform(0.5, 0.5),                 # 0.5-1.0
    'colsample_bytree': uniform(0.5, 0.5),          # 0.5-1.0
    'gamma': uniform(0, 5),                         # min split loss
    'reg_alpha': uniform(0, 1),                     # L1 regularization
    'reg_lambda': uniform(0, 2),                    # L2 regularization
    'min_child_weight': randint(1, 10)              # minimum sum of instance weight
}

In [4]:
all_results = []

for name, (train_path, test_path) in datasets.items():
    print(f"Training: {name}")
    
    # Data load
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]
    
    # XGBoost model
    xgb = XGBClassifier(
        random_state=42,
        eval_metric='auc'  
    )

    
    # Random Search
    random_search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_uniform,
        n_iter=100,                    
        scoring='roc_auc',
        cv=5,                          
        random_state=42,
        n_jobs=-1
    )
    
    # Fit
    random_search.fit(X_train, y_train)
    
    # Result
    cv_results = pd.DataFrame(random_search.cv_results_)
    
    # Testing on test sets
    for i, params in enumerate(random_search.cv_results_['params']):
        # Training again on parameters from random_search.cv_results_ :(
        model = XGBClassifier(
            random_state=42,
            eval_metric='auc',
            **params
        )
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)
        
        all_results.append({
            "dataset": name,
            "params": params,
            "cv_roc_auc": cv_results.loc[i, 'mean_test_score'],
            "test_roc_auc": test_auc
        })
    
#Results
results_df = pd.DataFrame(all_results)

Training: heart
Training: diabetes
Training: cancer
Training: alzheimer


In [5]:
#Summary
for dataset in datasets.keys():
    dataset_results = results_df[results_df['dataset'] == dataset]
    best_idx = dataset_results['test_roc_auc'].idxmax()
    best_result = dataset_results.loc[best_idx]
    
    print(f"\n{dataset.upper()}:")
    print(f"  Best test AUC: {best_result['test_roc_auc']:.4f}")
    print(f"  CV AUC: {best_result['cv_roc_auc']:.4f}")
    print(f"  Parameters: {best_result['params']}")


HEART:
  Best test AUC: 0.8005
  CV AUC: 0.8026
  Parameters: {'colsample_bytree': 0.6774525952313604, 'gamma': 4.784004425632282, 'learning_rate': 0.20626327228304794, 'max_depth': 6, 'min_child_weight': 3, 'n_estimators': 416, 'reg_alpha': 0.08328441119525964, 'reg_lambda': 0.18340829451696217, 'subsample': 0.8012204629505595}

DIABETES:
  Best test AUC: 0.8411
  CV AUC: 0.8199
  Parameters: {'colsample_bytree': 0.8343216099622155, 'gamma': 4.646879945637929, 'learning_rate': 0.17146123897403964, 'max_depth': 10, 'min_child_weight': 3, 'n_estimators': 222, 'reg_alpha': 0.7694929331919369, 'reg_lambda': 0.37408749711504674, 'subsample': 0.6618396182021218}

CANCER:
  Best test AUC: 0.8721
  CV AUC: 0.8662
  Parameters: {'colsample_bytree': 0.5157145928433671, 'gamma': 3.182052056318902, 'learning_rate': 0.10116323451213473, 'max_depth': 6, 'min_child_weight': 5, 'n_estimators': 456, 'reg_alpha': 0.6044173792778172, 'reg_lambda': 1.0796821826033463, 'subsample': 0.6015306123673847}

A

In [6]:
#Finding the best set of hyperparameters for each dataset 
best_per_dataset = (
    results_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False]).groupby("dataset", as_index=False).first()
)
params_df = best_per_dataset["params"].apply(pd.Series)

# Creating new set of hyperparameters from all datasets
mean_params = params_df.mean()

mean_params_dict = mean_params.to_dict()
for param in ["max_depth", "min_child_weight", "n_estimators"]:
    mean_params_dict[param] = int(round(mean_params_dict[param]))
mean_results = []

#Training with new hyperparameters on all datasets
for name, (train_path, test_path) in datasets.items():
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = XGBClassifier(
            random_state=42,
            eval_metric='auc',
            **mean_params_dict
        )
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    mean_auc = roc_auc_score(y_test, y_proba)

    mean_results.append({
        "dataset": name,
        "star_test_roc_auc": mean_auc
    })

mean_df = pd.DataFrame(mean_results)

In [7]:
# Creating new dataframe with all results

# Star means that this set of hyperparameters is a mean from the best 4 sets of hyperparameters, one for each set
params_df = results_df['params'].apply(pd.Series)
results_df = pd.concat([results_df.drop('params', axis=1), params_df], axis=1)

results_col = results_df[['cv_roc_auc','test_roc_auc']]
results_df = pd.concat([results_df.drop(['cv_roc_auc','test_roc_auc'],axis=1),results_col], axis=1)

results_df = results_df.merge(mean_df, on='dataset')
results_df['diff_from_star'] = results_df['star_test_roc_auc'] - results_df['test_roc_auc']
results_df

,dataset,colsample_bytree,gamma,learning_rate,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,subsample,cv_roc_auc,test_roc_auc,star_test_roc_auc,diff_from_star
0,heart,0.687270,4.753572,0.222278,7.0,7.0,171.0,0.155995,0.116167,0.933088,0.807082,0.784046,0.774757,-0.009290
1,heart,0.800558,3.540363,0.015970,4.0,8.0,463.0,0.212339,0.363650,0.591702,0.810297,0.787052,0.774757,-0.012296
2,heart,0.652121,2.623782,0.135264,3.0,3.0,413.0,0.514234,1.184829,0.523225,0.807207,0.775106,0.774757,-0.000349
3,heart,0.803772,0.852621,0.028865,6.0,9.0,365.0,0.563288,0.770833,0.507983,0.809995,0.777394,0.774757,-0.002638
4,heart,0.615447,1.205127,0.208146,14.0,8.0,480.0,0.173365,0.782121,0.591118,0.798637,0.737442,0.774757,0.037314
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,alzheimer,0.689411,2.285001,0.185148,10.0,5.0,206.0,0.229251,1.444505,0.860018,0.871094,0.852405,0.856776,0.004371
396,alzheimer,0.820574,3.469742,0.167390,13.0,5.0,197.0,0.181598,1.816901,0.791696,0.869915,0.856050,0.856776,0.000726
397,alzheimer,0.700426,2.310029,0.284712,14.0,8.0,495.0,0.100795,0.512031,0.863048,0.866920,0.853842,0.856776,0.002934
398,alzheimer,0.796481,0.511063,0.276438,7.0,8.0,342.0,0.707239,0.305078,0.788144,0.852435,0.836691,0.856776,0.020085


In [8]:
results_df.to_csv("Results/xgboost_uniform.csv", index=False)

In [9]:
# Best parameters for each dataset
best_per_dataset = (
    results_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
    .drop(['star_test_roc_auc', 'diff_from_star'], axis=1)
)

# STAR row
mean_row = {
    "dataset": "STAR",
    **mean_params_dict,
    "cv_roc_auc": 0,
    "test_roc_auc": mean_df["star_test_roc_auc"].mean()
}

summary_df = pd.concat([best_per_dataset, pd.DataFrame([mean_row])], ignore_index=True)

In [10]:
summary_df.to_csv("Results/xgboost_uniform_summary.csv", index=False)

# 2. Bayesian

In [11]:
# import pandas as pd
# import numpy as np
# from xgboost import XGBClassifier
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.metrics import roc_auc_score


In [12]:
# Search space for Bayesian Optimization
search_spaces = {
    'max_depth': Integer(3, 15),
    'learning_rate': Real(0.01, 0.3, prior='uniform'),
    'n_estimators': Integer(50, 500),
    'subsample': Real(0.5, 1.0, prior='uniform'),
    'colsample_bytree': Real(0.5, 1.0, prior='uniform'),
    'gamma': Real(0, 5, prior='uniform'),
    'reg_alpha': Real(0, 1, prior='uniform'),
    'reg_lambda': Real(0, 2, prior='uniform'),
    'min_child_weight': Integer(1, 10)
}


In [13]:
all_results_2 = []

for name, (train_path, test_path) in datasets.items():
    print(f"Training: {name}")
    
    # Data load
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]
    
    # XGBoost model
    xgb = XGBClassifier(
        random_state=42,
        eval_metric='auc'
    )
    
    # Bayesian Search
    bayes_search = BayesSearchCV(
        estimator=xgb,
        search_spaces=search_spaces,
        n_iter=100,
        scoring='roc_auc',
        cv=5,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    
    # Fit
    bayes_search.fit(X_train, y_train)
    
    # Result
    cv_results = pd.DataFrame(bayes_search.cv_results_)
 
    # Testing on test sets
    for i, params in enumerate(bayes_search.cv_results_['params']):
        # Training again on parameters from bayes_search.cv_results_
        model = XGBClassifier(
            random_state=42,
            eval_metric='auc',
            **params
        )
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)
        
        all_results_2.append({
            "dataset": name,
            "params": params,
            "cv_roc_auc": cv_results.loc[i, 'mean_test_score'],
            "test_roc_auc": test_auc
        })

# Results
results_2_df = pd.DataFrame(all_results_2)

Training: heart
Training: diabetes
Training: cancer
Training: alzheimer


In [14]:
# Summary
for dataset in datasets.keys():
    dataset_results = results_2_df[results_2_df['dataset'] == dataset]
    best_idx = dataset_results['test_roc_auc'].idxmax()
    best_result = dataset_results.loc[best_idx]
    
    print(f"\n{dataset.upper()}:")
    print(f"  Best test AUC: {best_result['test_roc_auc']:.4f}")
    print(f"  CV AUC: {best_result['cv_roc_auc']:.4f}")
    print(f"  Parameters: {best_result['params']}")



HEART:
  Best test AUC: 0.7965
  CV AUC: 0.8060
  Parameters: OrderedDict({'colsample_bytree': 0.6706960500327418, 'gamma': 4.713778166133631, 'learning_rate': 0.08517901029151521, 'max_depth': 7, 'min_child_weight': 10, 'n_estimators': 136, 'reg_alpha': 0.9360021654910516, 'reg_lambda': 1.95449583379019, 'subsample': 0.5295593162788603})

DIABETES:
  Best test AUC: 0.8423
  CV AUC: 0.8229
  Parameters: OrderedDict({'colsample_bytree': 0.5544190166660399, 'gamma': 5.0, 'learning_rate': 0.3, 'max_depth': 6, 'min_child_weight': 1, 'n_estimators': 151, 'reg_alpha': 0.3350022393249654, 'reg_lambda': 2.0, 'subsample': 0.5})

CANCER:
  Best test AUC: 0.8705
  CV AUC: 0.8688
  Parameters: OrderedDict({'colsample_bytree': 0.7048458637952003, 'gamma': 4.132791592839754, 'learning_rate': 0.07548543039370718, 'max_depth': 15, 'min_child_weight': 1, 'n_estimators': 326, 'reg_alpha': 0.0, 'reg_lambda': 1.947441430642152, 'subsample': 0.7699462691794106})

ALZHEIMER:
  Best test AUC: 0.8675
  CV AU

In [15]:
# Finding the best set of hyperparameters for each dataset
best_per_dataset_2 = (
    results_2_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False]).groupby("dataset", as_index=False).first()
)
params_2_df = best_per_dataset_2["params"].apply(pd.Series)

# Creating new set of hyperparameters from all datasets
mean_params_2 = params_2_df.mean()

mean_params_dict_2 = mean_params_2.to_dict()
for param in ["max_depth", "min_child_weight", "n_estimators"]:
    mean_params_dict_2[param] = int(round(mean_params_dict_2[param]))
mean_results = []

# Training with new hyperparameters on all datasets
for name, (train_path, test_path) in datasets.items():
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = XGBClassifier(
        random_state=42,
        eval_metric='auc',
        **mean_params_dict_2
    )
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    mean_auc = roc_auc_score(y_test, y_proba)

    mean_results.append({
        "dataset": name,
        "star_test_roc_auc": mean_auc
    })

mean_2_df = pd.DataFrame(mean_results)


In [16]:
# Creating new dataframe with all results
params_2_df = results_2_df['params'].apply(pd.Series)
results_2_df = pd.concat([results_2_df.drop('params', axis=1), params_2_df], axis=1)

results_col_2 = results_2_df[['cv_roc_auc','test_roc_auc']]
results_2df = pd.concat([results_2_df.drop(['cv_roc_auc','test_roc_auc'],axis=1),results_col_2], axis=1)

results_2_df = results_2_df.merge(mean_2_df, on='dataset')
results_2_df['diff_from_star'] = results_2_df['star_test_roc_auc'] - results_2_df['test_roc_auc']
results_2_df

,dataset,cv_roc_auc,test_roc_auc,colsample_bytree,gamma,learning_rate,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,subsample,star_test_roc_auc,diff_from_star
0,heart,0.801913,0.785191,0.705052,3.638629,0.280532,7.0,7.0,236.0,0.350931,1.479008,0.652232,0.783911,-0.001280
1,heart,0.810345,0.786820,0.918694,4.416576,0.097989,14.0,9.0,78.0,0.138309,0.707175,0.817865,0.783911,-0.002909
2,heart,0.805958,0.785734,0.722416,4.593613,0.040409,8.0,3.0,254.0,0.155448,1.503105,0.778670,0.783911,-0.001823
3,heart,0.799162,0.777433,0.906198,0.859358,0.183434,13.0,6.0,93.0,0.755801,1.745261,0.955964,0.783911,0.006478
4,heart,0.806067,0.759241,0.899777,2.190146,0.162720,12.0,9.0,373.0,0.424178,1.300568,0.676542,0.783911,0.024669
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,alzheimer,0.874836,0.860133,0.509042,1.775255,0.012065,12.0,8.0,459.0,0.916225,1.977802,0.966710,0.851119,-0.009014
396,alzheimer,0.877846,0.861555,0.843979,4.882500,0.010447,8.0,10.0,466.0,0.205658,1.975438,0.649383,0.851119,-0.010436
397,alzheimer,0.874722,0.857063,0.539447,2.100412,0.293890,10.0,9.0,55.0,0.566955,0.176318,0.974721,0.851119,-0.005944
398,alzheimer,0.880946,0.851709,0.500000,5.000000,0.300000,5.0,1.0,245.0,1.000000,0.559118,0.795474,0.851119,-0.000590


In [17]:
results_2_df.to_csv("Results/xgboost_bayes.csv", index=False)

In [18]:
# Best parameters for each dataset
best_per_dataset_2 = (
    results_2_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
    .drop(['star_test_roc_auc', 'diff_from_star'], axis=1)
)

# STAR row
mean_row_2 = {
    "dataset": "STAR",
    **mean_params_dict_2,
    "cv_roc_auc": 0,
    "test_roc_auc": mean_2_df["star_test_roc_auc"].mean()
}

summary_2_df = pd.concat([best_per_dataset_2, pd.DataFrame([mean_row_2])], ignore_index=True)

In [19]:
summary_2_df.to_csv("Results/xgboost_bayes_summary.csv", index=False)